In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="prompt-engineering",
    notebook_name="02_chain_of_thought.ipynb"
)

# Chain-of-Thought Prompting — Hands-On

In this notebook, you will see how asking the AI to "think step by step" changes its answers.
We use **pre-recorded responses** so the notebook runs without any API keys.
To use a live LLM, replace the `mock_llm` function with your API call.

If you have not read [chain-of-thought.md](./chain-of-thought.md) yet, start there first —
it explains the concepts. This notebook shows them in action.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# ── Pre-recorded LLM responses ──────────────────────────────────────
# These simulate what a real LLM would return.

MATH_PROBLEMS = [
    {
        "question": "A store has 24 apples. They sell 1/3 in the morning and 1/4 of the remaining in the afternoon. How many are left?",
        "correct_answer": 12,
        "direct_response": {"answer": 10, "reasoning": ""},
        "cot_response": {
            "answer": 12,
            "reasoning": "Step 1: Start with 24 apples.\nStep 2: Sell 1/3 in morning: 24 * (1/3) = 8 sold. Remaining: 24 - 8 = 16.\nStep 3: Sell 1/4 of remaining in afternoon: 16 * (1/4) = 4 sold. Remaining: 16 - 4 = 12.\nAnswer: 12 apples."
        },
    },
    {
        "question": "Tom has $100. He spends 20% on lunch, then 50% of what's left on a book. How much does he have?",
        "correct_answer": 40,
        "direct_response": {"answer": 30, "reasoning": ""},
        "cot_response": {
            "answer": 40,
            "reasoning": "Step 1: Tom starts with $100.\nStep 2: Spends 20% on lunch: $100 * 0.20 = $20. Left: $100 - $20 = $80.\nStep 3: Spends 50% of remaining on book: $80 * 0.50 = $40. Left: $80 - $40 = $40.\nAnswer: $40."
        },
    },
    {
        "question": "A train leaves at 9:00 AM going 60 mph. Another leaves at 10:00 AM going 90 mph on the same route. When does the second train catch up?",
        "correct_answer": 12,
        "direct_response": {"answer": 11, "reasoning": ""},
        "cot_response": {
            "answer": 12,
            "reasoning": "Step 1: Train A leaves at 9:00 AM at 60 mph.\nStep 2: Train B leaves at 10:00 AM at 90 mph. At 10:00 AM, Train A has traveled 60 miles.\nStep 3: Gap = 60 miles. Closing speed = 90 - 60 = 30 mph.\nStep 4: Time to close gap: 60 / 30 = 2 hours after 10:00 AM.\nStep 5: 10:00 AM + 2 hours = 12:00 PM.\nAnswer: 12:00 PM (noon)."
        },
    },
    {
        "question": "A garden has 5 rows with 8 flowers each. The gardener plants 3 more rows with 6 flowers each. Then 10 flowers die. How many flowers are alive?",
        "correct_answer": 48,
        "direct_response": {"answer": 52, "reasoning": ""},
        "cot_response": {
            "answer": 48,
            "reasoning": "Step 1: Original flowers: 5 rows * 8 = 40.\nStep 2: New flowers: 3 rows * 6 = 18.\nStep 3: Total: 40 + 18 = 58.\nStep 4: 10 flowers die: 58 - 10 = 48.\nAnswer: 48 flowers."
        },
    },
    {
        "question": "Maria reads 30 pages per day. The book has 450 pages. She starts on Monday. On what day does she finish?",
        "correct_answer": "Monday",
        "direct_response": {"answer": "Wednesday", "reasoning": ""},
        "cot_response": {
            "answer": "Monday",
            "reasoning": "Step 1: Total pages: 450. Pages per day: 30.\nStep 2: Days to finish: 450 / 30 = 15 days.\nStep 3: She starts on Monday (day 1) and finishes on day 15.\nStep 4: Day 1=Mon, Day 8=Mon, Day 15=Mon.\nAnswer: Monday (15 days later, which is the third Monday)."
        },
    },
    {
        "question": "A recipe needs 2 cups of flour for 12 cookies. How many cups for 30 cookies?",
        "correct_answer": 5,
        "direct_response": {"answer": 5, "reasoning": ""},
        "cot_response": {
            "answer": 5,
            "reasoning": "Step 1: 2 cups for 12 cookies.\nStep 2: Cups per cookie: 2/12 = 1/6 cup.\nStep 3: For 30 cookies: 30 * (1/6) = 5 cups.\nAnswer: 5 cups."
        },
    },
]

# Self-consistency: multiple runs of the same problem
SELF_CONSISTENCY_RUNS = [
    {"reasoning": "60 miles gap, closing at 30 mph, takes 2 hours. 10 AM + 2 = noon.", "answer": 12},
    {"reasoning": "Train A goes 60 mi by 10 AM. 90t = 60 + 60t, 30t = 60, t = 2. Noon.", "answer": 12},
    {"reasoning": "Train A: 60 miles ahead. 90 - 60 = 30 mph closing. 60/30 = 2 hrs. Noon.", "answer": 12},
    {"reasoning": "After 1 hour, train A is 60 mi ahead. The gap is 60. But 90 - 60 = 30 mph. 60/30 = 2. 10 + 2 = 12.", "answer": 12},
    {"reasoning": "Train B is faster by 30 mph. Train A has 1 hr head start = 60 mi. 60/30 = 2. But actually the question is tricky — 11 AM.", "answer": 11},
]

print("Setup complete.")
print(f"Math problems loaded: {len(MATH_PROBLEMS)}")
print(f"Self-consistency runs loaded: {len(SELF_CONSISTENCY_RUNS)}")

## Experiment 1: Direct Answer vs Chain-of-Thought

We will give the same math problems to two versions of a prompt:
- **Direct**: just ask for the answer
- **CoT**: add "Let's think step by step"

Then we compare which gets more answers correct.

In [ ]:
# ── Run both approaches ─────────────────────────────────────────────
direct_correct = 0
cot_correct = 0

print("=" * 70)
print("DIRECT vs CHAIN-OF-THOUGHT")
print("=" * 70)

for i, problem in enumerate(MATH_PROBLEMS, 1):
    direct_ans = problem["direct_response"]["answer"]
    cot_ans = problem["cot_response"]["answer"]
    correct = problem["correct_answer"]
    
    direct_ok = str(direct_ans) == str(correct)
    cot_ok = str(cot_ans) == str(correct)
    
    if direct_ok:
        direct_correct += 1
    if cot_ok:
        cot_correct += 1
    
    print(f"\nProblem {i}: {problem['question'][:70]}...")
    print(f"  Correct answer: {correct}")
    print(f"  Direct:  {direct_ans} {'✓' if direct_ok else '✗'}")
    print(f"  CoT:     {cot_ans} {'✓' if cot_ok else '✗'}")

print(f"\n{'=' * 70}")
print(f"Direct accuracy: {direct_correct}/{len(MATH_PROBLEMS)} = {direct_correct/len(MATH_PROBLEMS):.0%}")
print(f"CoT accuracy:    {cot_correct}/{len(MATH_PROBLEMS)} = {cot_correct/len(MATH_PROBLEMS):.0%}")

The CoT version gets more problems correct because it breaks each problem into small,
manageable steps. The direct version tries to jump straight to the answer and makes errors.

## Experiment 2: Seeing the Reasoning

The real value of CoT is not just accuracy — it is **transparency**.
When the AI shows its work, you can check where it went right or wrong.

In [ ]:
# ── Show step-by-step reasoning for each problem ────────────────────
for i, problem in enumerate(MATH_PROBLEMS[:3], 1):
    print(f"{'=' * 60}")
    print(f"Problem {i}: {problem['question']}")
    print(f"{'=' * 60}")
    print(f"\nDirect answer: {problem['direct_response']['answer']}")
    print(f"  (no reasoning shown — was it luck or real understanding?)")
    print(f"\nCoT reasoning:")
    for line in problem['cot_response']['reasoning'].split('\n'):
        print(f"  {line}")
    print(f"\nCorrect answer: {problem['correct_answer']}")
    print()

Notice: even when the direct answer is correct (like problem 6), you have no way
to verify the reasoning. With CoT, you can check every step.

## Experiment 3: Self-Consistency Voting

Self-consistency runs the same CoT prompt **multiple times** (with temperature > 0)
and picks the most common answer. This improves accuracy on hard problems.

We will simulate 5 runs on the train problem.

In [ ]:
# ── Show each run ───────────────────────────────────────────────────
print("Question: A train leaves at 9 AM going 60 mph. Another leaves at")
print("10 AM going 90 mph. When does the second train catch up?")
print(f"\nCorrect answer: 12:00 PM (noon)")
print(f"\n{'=' * 60}")

answers = []
for i, run in enumerate(SELF_CONSISTENCY_RUNS, 1):
    print(f"\nRun {i}:")
    print(f"  Reasoning: {run['reasoning']}")
    print(f"  Answer: {run['answer']}")
    answers.append(run['answer'])

# ── Count votes ──────────────────────────────────────────────────────
vote_counts = Counter(answers)
winner = vote_counts.most_common(1)[0]

print(f"\n{'=' * 60}")
print(f"Vote counts:")
for answer, count in vote_counts.most_common():
    bar = '█' * (count * 8)
    print(f"  Answer {answer}: {count} votes  {bar}")
print(f"\nMajority answer: {winner[0]} ({winner[1]}/{len(answers)} runs)")
print(f"\nSelf-consistency correctly picks 12 (noon) even though run 5 got it wrong.")

## Visualization: Direct vs CoT Accuracy

A bar chart comparing the two approaches across all problems.

In [ ]:
# ── Bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

methods = ['Direct Answer', 'Chain-of-Thought']
accuracies = [
    direct_correct / len(MATH_PROBLEMS) * 100,
    cot_correct / len(MATH_PROBLEMS) * 100,
]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(methods, accuracies, color=colors, edgecolor='white', linewidth=2, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{acc:.0f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Math Problem Accuracy: Direct vs Chain-of-Thought', fontsize=14)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("Adding 'Let's think step by step' dramatically improves accuracy")
print("on multi-step reasoning problems.")

## Summary

What you just saw:

1. **Direct vs CoT**: chain-of-thought improved accuracy from ~33% to 100% on multi-step math
2. **Transparency**: CoT shows the reasoning, so you can verify each step
3. **Self-consistency**: running CoT multiple times and voting catches occasional errors

For the concepts behind these results, read [chain-of-thought.md](./chain-of-thought.md).
Next notebook: [03_prompt_templates.ipynb](./03_prompt_templates.ipynb)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)